In [1]:
import sys
sys.path.insert(0, '../')

# **0. 匯入套件和載入資料**

In [2]:
# === 效能評估計時器：初始化 ===
# 這個 cell 在執行批次回測之前先準備好計時與規格紀錄。
# 唯一需要切換的設定：USE_CACHE
#   - True  : 啟用所有快取
#   - False : 關閉所有快取（與 True 版本作為快取效益對照）

import time
import json as _json_for_timing
from datetime import datetime
from pathlib import Path as _Path_for_timing

USE_CACHE = True   # ← 切換這裡：True 跑啟用快取版、False 跑無快取版

timings = {}            # 各階段耗時 (秒)
strategy_count = {}     # 各階段策略數量
hardware_info = {       # 硬體規格自動偵測
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "USE_CACHE": USE_CACHE,
}

try:
    import platform
    hardware_info["os"] = platform.platform()
    hardware_info["python"] = platform.python_version()
    hardware_info["machine"] = platform.machine()
    hardware_info["processor"] = platform.processor() or "unknown"
except Exception as e:
    hardware_info["platform_error"] = str(e)

try:
    import psutil
    hardware_info["cpu_count_physical"] = psutil.cpu_count(logical=False)
    hardware_info["cpu_count_logical"] = psutil.cpu_count(logical=True)
    hardware_info["ram_gb"] = round(psutil.virtual_memory().total / (1024**3), 1)
    hardware_info["cpu_freq_max_mhz"] = round(psutil.cpu_freq().max, 0) if psutil.cpu_freq() else None
except ImportError:
    print("[Timer] 提示：未安裝 psutil，CPU/RAM 規格將無法自動偵測。")
    print("        如要自動偵測，請執行：pip install psutil")
except Exception as e:
    hardware_info["psutil_error"] = str(e)

total_start = time.time()
print(f"[Timer] 全域計時器啟動")
print(f"[Timer] USE_CACHE = {USE_CACHE}")
print(f"[Timer] 硬體規格：")
for k, v in hardware_info.items():
    print(f"        {k}: {v}")


[Timer] 全域計時器啟動
[Timer] USE_CACHE = True
[Timer] 硬體規格：
        timestamp: 2026-07-01T21:07:05
        USE_CACHE: True
        os: Windows-10-10.0.26200-SP0
        python: 3.10.11
        machine: AMD64
        processor: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
        cpu_count_physical: 20
        cpu_count_logical: 28
        ram_gb: 63.7
        cpu_freq_max_mhz: 2100.0


In [3]:
import sys
sys.path.insert(0, '../')


In [4]:
from get_data import Data
import backtest
from combinations import sim_conditions
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

In [5]:
data=Data()

In [6]:
'''
下載資料
1. price:股價相關
2. report:財報相關
'''
data.get("price:close")

company_symbol,1101,1102,1103,1104,1108,1109,1110,1201,1203,1210,...,9944,9945,9946,9949,9950,9951,9955,9958,9960,9962
date,,,,,,,,,,,,,,,,,,,,,
2000-01-04,7.5067,5.3558,5.5019,2.6105,5.4752,5.4740,6.2886,6.5692,6.4118,1.7259,...,NaN,1.5950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,7.7020,5.7189,5.5304,2.6507,5.6920,5.5840,6.2573,7.0125,6.4685,1.7978,...,NaN,1.6852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,7.3766,5.7734,5.7584,2.7042,5.6920,5.7159,6.4450,7.4961,6.6387,1.8377,...,NaN,1.7605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,7.3549,5.7734,5.8155,2.6908,5.6920,5.7379,6.4450,8.0200,6.6955,1.8457,...,NaN,1.7605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-10,7.5935,5.8097,5.9580,2.6908,5.6107,5.6719,6.3512,8.5439,6.8941,1.8537,...,NaN,1.7605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-25,34.3500,40.3000,18.8000,29.3500,15.7000,18.1500,18.7500,19.0000,53.2000,57.7000,...,20.15,36.2500,20.15,17.60,22.8,75.3,24.20,168.0,27.30,18.45
2023-12-26,34.5000,40.5500,18.8000,29.4000,15.7500,18.1000,18.5500,19.0500,53.5000,57.4000,...,20.20,36.4000,20.15,18.05,22.6,76.8,24.20,167.0,27.25,18.30
2023-12-27,34.6500,40.9000,18.9000,29.5500,15.8000,18.1000,18.7500,19.0500,54.7000,57.3000,...,20.15,36.8000,20.40,17.90,22.4,75.8,24.45,167.0,27.10,18.20


In [7]:
data.get("report:REVENUE")

company_symbol


# **2. 依據策略將股票分群**

## **2.0 讀入 JSON 檔案**

In [8]:
data.raw_report_data["factor_name"].unique() 

array(['EPS', 'PE', 'EV_EBITDA', 'EV_S', 'FCF_P', 'CROIC', 'FCF_OI',
       'ROE', 'ROIC', 'PB', 'PS', 'P_IC', 'OCF_E', 'MOM'], dtype=object)

In [9]:
data.get("report:REVENUE")

company_symbol


In [10]:
# ================================================================
# 【正確性與效能優化】資料對齊、快取、邊界檢查
# ================================================================
import warnings
from functools import lru_cache

# === 1. 統一的資料對齊函式（確保 report 類資料與交易日對齊，避免未來資訊） ===
def align_to_trading_days(df: pd.DataFrame, price_index: pd.DatetimeIndex, 
                          field_name: str = "", warn_on_forward_fill: bool = True) -> pd.DataFrame:
    """
    將任意 DataFrame 的索引對齊到 price:close 的交易日索引。
    
    關鍵規則：
    - report 類資料（已透過 adjust_index_of_report 調整到可得日）必須用 ffill 延續，
      確保「在可得日當天及之後」才能使用該財報資訊，避免 look-ahead bias。
    - price 類資料通常已與交易日對齊，只需 reindex 即可。
    
    Args:
        df: 要對齊的 DataFrame（index 應為日期）
        price_index: price:close 的 DatetimeIndex（交易日基準）
        field_name: 欄位名稱（用於警告訊息）
        warn_on_forward_fill: 是否在 ffill 時發出警告（提醒檢查是否合理）
    
    Returns:
        對齊後的 DataFrame（index = price_index，columns 不變）
    """
    if df is None or df.empty:
        return df
    
    # 確保 df.index 是 DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df.index = pd.to_datetime(df.index, errors="coerce")
        if df.index.isna().any():
            bad_cnt = df.index.isna().sum()
            warnings.warn(f"[align_to_trading_days] {field_name} 有 {bad_cnt} 筆日期無法解析為 datetime，將被移除。")
            df = df.loc[df.index.notna()]
    
    # 排序索引
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    
    # 判斷是否為 report 類（通常頻率較低，且已調整到可得日）
    is_report_like = False
    if field_name.startswith("report:"):
        is_report_like = True
        # 檢查：report 資料的日期應該「不晚於」對應的交易日
        # （因為 adjust_index_of_report 已經把季報結束日推到可得日）
        if len(df.index) > 0 and len(price_index) > 0:
            earliest_report = df.index.min()
            earliest_price = price_index.min()
            if earliest_report > earliest_price:
                warnings.warn(
                    f"[align_to_trading_days] {field_name} 最早日期 ({earliest_report.date()}) "
                    f"晚於價格最早交易日 ({earliest_price.date()})，請檢查 adjust_index_of_report 規則。"
                )
    
    # 對齊到交易日：report 類用 ffill（確保可得日當天及之後才可用），price 類也用 ffill（延續最後已知值）
    aligned = df.reindex(price_index, method="ffill")
    
    if warn_on_forward_fill and is_report_like and len(df.index) < len(price_index) * 0.3:
        # 如果 report 資料點數遠少於交易日，可能是正常的（季報 vs 日頻）
        pass  # 不警告，這是預期行為
    
    return aligned

# === 2. 快取對齊後的資料（避免重複對齊） ===
_aligned_cache = {}  # {field_name: aligned_df}

def get_aligned_field(data, field_name: str, price_index: pd.DatetimeIndex) -> pd.DataFrame:
    """
    取得對齊到交易日的欄位資料（帶快取）。
    
    使用方式：
        price_idx = data.get("price:close").index
        roe_aligned = get_aligned_field(data, "report:roe", price_idx)
    """
    cache_key = f"{field_name}_{id(price_index)}"
    if cache_key in _aligned_cache:
        return _aligned_cache[cache_key]
    
    raw_df = data.get(field_name)
    if raw_df is None:
        return None
    
    aligned = align_to_trading_days(raw_df, price_index, field_name=field_name)
    _aligned_cache[cache_key] = aligned
    return aligned

# === 3. 建立基準交易日索引（一次性取得，後續所有對齊都以此為準） ===
def get_trading_day_index(data) -> pd.DatetimeIndex:
    """取得 price:close 的交易日索引（作為所有資料對齊的基準）。"""
    price_close = data.get("price:close")
    if price_close is None or price_close.empty:
        raise ValueError("無法取得 price:close，請先確保資料已載入。")
    idx = price_close.index
    if not isinstance(idx, pd.DatetimeIndex):
        idx = pd.to_datetime(idx)
    return idx

# === 4. 清除快取（當需要重新載入資料時呼叫） ===
def clear_alignment_cache():
    """清除對齊快取（例如更換資料集時）。"""
    global _aligned_cache
    _aligned_cache.clear()
    print("已清除對齊快取。")

print("資料對齊與快取函式已載入。")

資料對齊與快取函式已載入。


In [11]:
import json
from pathlib import Path
from condition_factory import build_conditions

def load_P1_P3_from_json(json_path: str):
    with open(json_path, "r", encoding="utf-8") as f:
        config = json.load(f)
    P1_conditions = build_conditions(config.get("P1", {}))
    P3_conditions = build_conditions(config.get("P3", {}))
    return P1_conditions, P3_conditions

# === 多個 JSON 一次載入 ===
# json_files = [
#     "condition_config.json",
#     "test_1.json"
# ]

json_files = [
    "fcv_experiment_spec.json",
    #"condition_config_test_10.json"
]

In [12]:
# === 跳過已跑過的 JSON：檢查結果目錄是否已存在 ===
from pathlib import Path

PICKLE_DIR = Path("results_pickle")
ART_DIR = Path("results_artifacts")

# 強制重跑所有 JSON（即使結果已存在）。改設定後要重跑時，把這個切成 True。
FORCE_RERUN = True


def has_existing_results(label: str) -> bool:
    """檢查指定 label 的回測結果是否已存在於 PICKLE_DIR 或 ART_DIR。"""
    pickle_path = PICKLE_DIR / label
    art_path = ART_DIR / label
    pickle_ok = pickle_path.exists() and any(pickle_path.iterdir())
    art_ok = art_path.exists() and any(art_path.iterdir())
    return pickle_ok or art_ok


if FORCE_RERUN:
    print("FORCE_RERUN=True,所有 JSON 都會重跑。")
else:
    pending = []
    skipped = []
    for jp in json_files:
        label = Path(jp).stem
        if has_existing_results(label):
            skipped.append(label)
            print(f"[Skip] {label} 的結果已存在")
        else:
            pending.append(jp)
    
    print(f"\n總共 {len(json_files)} 個 JSON,跳過 {len(skipped)} 個,"
          f"要跑 {len(pending)} 個。")
    json_files = pending

FORCE_RERUN=True,所有 JSON 都會重跑。


In [13]:
# === Mask 計算快取（避免重複計算相同條件） ===
_mask_cache = {}  # {(field_name, cond_name): mask_df}

def build_masks(conditions: list, field_cache: dict, price_index: pd.DatetimeIndex = None, prefix="") -> dict:
    """
    建立條件遮罩（帶對齊檢查與快取）。
    
    Args:
        conditions: 條件列表（每個元素有 "name", "field", "cond"）
        field_cache: 欄位快取字典（key=field_name, value=DataFrame）
        price_index: 交易日索引（用於對齊檢查，若為 None 則跳過對齊）
        prefix: 前綴（用於日誌）
    
    Returns:
        {condition_name: mask_df} 字典
    """
    masks = {}
    
    # 若未提供 price_index，嘗試從 field_cache 中取得 price:close 的索引
    if price_index is None:
        price_close = field_cache.get("price:close")
        if price_close is not None and hasattr(price_close, 'index'):
            price_index = price_close.index
            if not isinstance(price_index, pd.DatetimeIndex):
                price_index = pd.to_datetime(price_index)
        else:
            warnings.warn(f"[build_masks] 未提供 price_index，將跳過對齊檢查。")
    
    for cond in conditions:
        field_name = cond["field"]
        cond_name = cond["name"]
        
        # 檢查快取（受全域 USE_CACHE 旗標控制）
        cache_key = (field_name, cond_name)
        if USE_CACHE and cache_key in _mask_cache:
            masks[cond_name] = _mask_cache[cache_key]
            continue
        
        # 取得原始資料
        df = field_cache.get(field_name)
        if df is None:
            print(f"{prefix}欄位 {field_name} 不存在，略過 {cond_name}")
            continue
        
        # 【正確性檢查】確保資料已對齊到交易日（特別是 report 類）
        if price_index is not None:
            # 檢查索引是否已對齊
            if not df.index.equals(price_index):
                # 若未對齊，進行對齊（使用統一的對齊函式）
                df_aligned = align_to_trading_days(df, price_index, field_name=field_name)
            else:
                df_aligned = df
        else:
            df_aligned = df
        
        # 計算遮罩
        try:
            mask = cond["cond"](df_aligned)
            
            # 【正確性檢查】確保 mask 的索引與交易日一致
            if price_index is not None and not mask.index.equals(price_index):
                warnings.warn(
                    f"[build_masks] {cond_name} 的 mask 索引未對齊到交易日。"
                    f"原始索引範圍: {mask.index.min()} ~ {mask.index.max()}, "
                    f"交易日範圍: {price_index.min()} ~ {price_index.max()}"
                )
                # 強制對齊
                mask = mask.reindex(price_index, method="ffill").fillna(False)
            
            masks[cond_name] = mask
            if USE_CACHE:
                _mask_cache[cache_key] = mask  # 快取結果
            
        except Exception as e:
            warnings.warn(f"[build_masks] 計算 {cond_name} 時發生錯誤: {e}")
            continue
    
    return masks

# === 清除 mask 快取 ===
def clear_mask_cache():
    """清除 mask 計算快取。"""
    global _mask_cache
    _mask_cache.clear()
    print("已清除 mask 快取。")


## **2.1 F 構面篩選**

In [14]:
from collections import defaultdict

def f_factor(P1_conditions, field_cache, price_index: pd.DatetimeIndex = None):
    """
    F 構面篩選（P1 + P2 組合），帶對齊檢查與快取。
    
    Args:
        P1_conditions: P1 條件列表
        field_cache: 欄位快取字典
        price_index: 交易日索引（用於對齊檢查）
    
    Returns:
        {f"{p1_name}__{p2_name}": mask_df} 字典
    """
    # 建立所有 P1 遮罩（使用優化後的 build_masks，帶對齊檢查）
    P1_masks = build_masks(P1_conditions, field_cache, price_index=price_index, prefix="P1")

    # 建立 P2 遮罩候選映射，排除 P1 條件的欄位
    P2_candidate_map = defaultdict(list)
    for p1_cond in P1_conditions:
        p1_name = p1_cond["name"]
        p1_field = p1_cond["field"]

        P2_candidate_map[p1_name].append(None)  # baseline
        for p2_cond in P1_conditions:
            if p2_cond["field"] == p1_field:
                continue
            P2_candidate_map[p1_name].append(p2_cond)

    p1p2_masks = {}

    # 執行 P1 + P2 條件組合
    for p1_cond in P1_conditions:
        p1_name = p1_cond["name"]
        p1_mask = P1_masks.get(p1_name)
        if p1_mask is None:
            continue

        for p2_cond in P2_candidate_map[p1_name]:
            if p2_cond is None:
                # P2 = 空集合 baseline
                final_mask = p1_mask.copy()
                p2_name = "None"
            else:
                p2_name = p2_cond["name"]
                p2_field = p2_cond["field"]
                p2_func = p2_cond["cond"]

                # 【效能優化】檢查是否已有快取的 P2 mask（受 USE_CACHE 控制）
                p2_cache_key = (p2_field, p2_name)
                if USE_CACHE and p2_cache_key in _mask_cache:
                    p2_mask = _mask_cache[p2_cache_key]
                else:
                    df = field_cache.get(p2_field)
                    if df is None:
                        print(f"P2 欄位 {p2_field} 不存在，略過 {p2_name}")
                        continue
                    
                    # 【正確性檢查】確保 P2 資料已對齊
                    if price_index is not None and not df.index.equals(price_index):
                        df = align_to_trading_days(df, price_index, field_name=p2_field)
                    
                    p2_mask = p2_func(df)
                    
                    # 【正確性檢查】確保 P2 mask 索引對齊
                    if price_index is not None and not p2_mask.index.equals(price_index):
                        p2_mask = p2_mask.reindex(price_index, method="ffill").fillna(False)
                    
                    if USE_CACHE:
                        _mask_cache[p2_cache_key] = p2_mask  # 快取 P2 mask
                
                final_mask = p1_mask & p2_mask
            
            # 【正確性檢查】確保最終 mask 索引一致
            if price_index is not None and not final_mask.index.equals(price_index):
                final_mask = final_mask.reindex(price_index, method="ffill").fillna(False)
            
            p1p2_masks[f"{p1_name}__{p2_name}"] = final_mask

            # 執行回測或資料統計
            count = final_mask.sum().sum()
            print(f"條件組合：P1 = {p1_name}，P2 = {p2_name}（符合筆數：{count}）")
        
    return p1p2_masks


## **2.2 C 構面篩選**

In [15]:
def c_factor(p1p2_masks, P3_conditions, field_cache, price_index: pd.DatetimeIndex = None):
    """
    C 構面篩選（在固定 F 下加入 C 條件），帶對齊檢查。
    
    Args:
        p1p2_masks: F 構面的 mask 字典（{f"{p1_name}__{p2_name}": mask_df}）
        P3_conditions: P3 條件列表
        field_cache: 欄位快取字典
        price_index: 交易日索引（用於對齊檢查）
    
    Returns:
        {f"{p1_name}__{p2_name}__{p3_name}": mask_df} 字典
    """
    # 建立所有 P3 遮罩（使用優化後的 build_masks，帶對齊檢查）
    P3_masks = build_masks(P3_conditions, field_cache, price_index=price_index, prefix="P3")
    final_masks = {}

    for combo_key, base_mask in p1p2_masks.items():
        # 【正確性檢查】確保 base_mask 索引對齊
        if price_index is not None and not base_mask.index.equals(price_index):
            base_mask = base_mask.reindex(price_index, method="ffill").fillna(False)
        
        P3_cond_list = [None] + list(P3_masks.items())  # baseline + 各 P3 條件
        for p3 in P3_cond_list:
            if p3 is None:
                # P3 = None（baseline）
                final_final_mask = base_mask.copy()
                combo3_key = f"{combo_key}__None"
            else:
                p3_name, p3_mask = p3
                combo3_key = f"{combo_key}__{p3_name}"
                
                # 【正確性檢查】確保 P3 mask 索引對齊
                if price_index is not None and not p3_mask.index.equals(price_index):
                    p3_mask = p3_mask.reindex(price_index, method="ffill").fillna(False)
                
                final_final_mask = base_mask & p3_mask
            
            # 【正確性檢查】確保最終 mask 索引一致
            if price_index is not None and not final_final_mask.index.equals(price_index):
                final_final_mask = final_final_mask.reindex(price_index, method="ffill").fillna(False)
            
            final_masks[combo3_key] = final_final_mask
            print(f"條件組合：P1+P2 = {combo_key}，P3 = {p3[0] if p3 else 'None'}（符合筆數：{final_final_mask.sum().sum()}）")
    
    return final_masks

In [16]:
# === V 構面（估值）：固定 P/E 條件，為每個策略產生 v0 / v1 版本 ===
# v0：不加任何估值限制（基準組）
# v1：當季 P/E 低於近四季平均，且高於近四季最低（略便宜但非深度折價）

import pandas as pd

# === V mask 快取 ===
_v_mask_cache = None
_v_mask_cache_key = None

def build_v_pe_mask(data, pe_field: str = "report:pe", window: int = 4,
                    price_index: pd.DatetimeIndex = None, daily_to_quarter: bool = True,
                    use_cache: bool = True):
    """
    建立 V 構面的 P/E 估值遮罩（帶對齊檢查與快取）。

    【正確性規則（避免 look-ahead）】
    - 近四季 mean/min 的 rolling 必須在「低頻序列（季/財報可得日序列）」上完成，再對齊回交易日；
      否則若先展平成日頻，rolling(window=4) 會變成 4 個交易日而不是 4 季。
    - 對齊到交易日使用 ffill，確保「在可得日當天及之後」才生效。

    Args:
        data: Data 物件
        pe_field: P/E 欄位名稱（預設 "report:pe"）
        window: rolling 視窗（預設 4，即近四季）
        price_index: 交易日索引（若為 None，會從 data.get("price:close") 取得）
        daily_to_quarter: 是否將日頻轉為季頻後再做近四季（預設 True）
        use_cache: 是否使用快取（預設 True）

    Returns:
        v1_mask: DataFrame（index=交易日，columns=股票，值為 True/False）
    """
    global _v_mask_cache, _v_mask_cache_key

    # 取得交易日索引
    if price_index is None:
        price_close = data.get("price:close")
        if price_close is None or getattr(price_close, "empty", True):
            raise ValueError("無法取得 price:close，請先確保資料已載入。")
        price_index = price_close.index

    if not isinstance(price_index, pd.DatetimeIndex):
        price_index = pd.to_datetime(price_index)

    # 檢查快取
    cache_key = (pe_field, window, id(price_index))
    if use_cache and _v_mask_cache is not None and _v_mask_cache_key == cache_key:
        return _v_mask_cache

    # 取得原始 P/E 資料
    pe_raw = data.get(pe_field)
    if pe_raw is None or getattr(pe_raw, "empty", True):
        raise ValueError(f"無法取得 {pe_field}，請檢查資料是否存在。")

    # 【關鍵修正】rolling 必須在低頻序列上做（先別對齊成日頻）
    pe_base = pe_raw.sort_index()
    if not isinstance(pe_base.index, pd.DatetimeIndex):
        pe_base.index = pd.to_datetime(pe_base.index)

    pe_for_rolling = pe_base

    # 若 P/E 是日頻（或太高頻），先轉為季頻（每季取最後一筆）
    if daily_to_quarter and len(pe_base.index) >= 10:
        median_gap = pe_base.index.to_series().diff().dropna().dt.days.median()
        if (median_gap is not None) and (median_gap < 40):
            # pandas 2.2 起 Q → QE 的別名更動方向；做相容 fallback
            try:
                pe_for_rolling = pe_base.resample("QE").last()  # QuarterEnd
            except Exception:
                pe_for_rolling = pe_base.resample("Q").last()

    # 在低頻序列上做「近四季」rolling
    pe_mean4_q = pe_for_rolling.rolling(window, min_periods=window).mean()
    pe_min4_q  = pe_for_rolling.rolling(window, min_periods=window).min()

    # v1：P/E 低於近四季平均，且高於近四季最低
    v1_q = (pe_for_rolling < pe_mean4_q) & (pe_for_rolling > pe_min4_q)

    # 對齊回交易日（用 ffill，確保「可得日當天及之後」才生效）
    v1 = v1_q.reindex(price_index, method="ffill").fillna(False)

    # 快取結果
    if use_cache:
        _v_mask_cache = v1
        _v_mask_cache_key = cache_key

    return v1

def expand_with_v(final_masks: dict, v_mask: pd.DataFrame,
                  suffix_v0: str = "__v0", suffix_v1: str = "__v1",
                  price_index: pd.DatetimeIndex = None):
    """
    將每個策略展開為 v0（不加估值）與 v1（加估值）兩個版本。
    """
    expanded = {}

    # 若未提供 price_index，嘗試從 v_mask 取得
    if price_index is None and hasattr(v_mask, "index"):
        price_index = v_mask.index

    for name, m in final_masks.items():
        # 【正確性檢查】確保 mask 索引對齊
        if price_index is not None and not m.index.equals(price_index):
            m = m.reindex(price_index, method="ffill").fillna(False)

        # v0：不加估值限制（原 mask）
        expanded[f"{name}{suffix_v0}"] = m.copy()

        # v1：加估值限制（mask & v_mask）
        v_mask_aligned = v_mask
        if price_index is not None and not v_mask.index.equals(price_index):
            v_mask_aligned = v_mask.reindex(price_index, method="ffill").fillna(False)

        expanded[f"{name}{suffix_v1}"] = (m & v_mask_aligned)

    return expanded

# === 清除 V mask 快取 ===
def clear_v_mask_cache():
    """清除 V mask 快取。"""
    global _v_mask_cache, _v_mask_cache_key
    _v_mask_cache = None
    _v_mask_cache_key = None
    print("已清除 V mask 快取。")


## **2.3 建立策略並回測**

In [17]:
from pathlib import Path
from io_persistence import save_all_for_label

final_masks_by_json = {}
report_collections_by_json = {}

PICKLE_DIR = Path("results_pickle")      # 新增：整包 pickle 存放處
ART_DIR    = Path("results_artifacts")   # 新增：輕量素材輸出處（每策略檔）

# === 【效能優化】一次性取得交易日索引，後續所有對齊都以此為準 ===
price_index = get_trading_day_index(data)
print(f"交易日索引已取得：{len(price_index)} 個交易日，範圍 {price_index.min().date()} ~ {price_index.max().date()}")

# === 【效能評估】階段 1：條件解析計時開始 ===
t_parse_start = time.time()

for jp in json_files:
    P1_conditions, P3_conditions = load_P1_P3_from_json(jp)
    label = Path(jp).stem
    print(f"\n=== {label} / P1 ===")
    for cond in P1_conditions:
        print(f"- {cond['name']}")
    print(f"\n=== {label} / P3 ===")
    for cond in P3_conditions:
        print(f"- {cond['name']}")

    # === 【效能優化】建立欄位快取（已對齊到交易日） ===
    field_cache = {}
    all_fields = {cond["field"] for cond in P1_conditions + P3_conditions}
    
    # 加入 price:close（若尚未存在）以確保後續對齊檢查可用
    if "price:close" not in all_fields:
        all_fields.add("price:close")
    
    for field in all_fields:
        raw_df = data.get(field)
        if raw_df is None:
            print(f"欄位 {field} 不存在，將跳過相關條件。")
            continue
        
        # 【正確性檢查】對齊到交易日（特別是 report 類資料）
        if field.startswith("report:"):
            aligned_df = align_to_trading_days(raw_df, price_index, field_name=field)
            field_cache[field] = aligned_df
        else:
            # price 類資料通常已對齊，但仍檢查一次
            if not raw_df.index.equals(price_index):
                aligned_df = align_to_trading_days(raw_df, price_index, field_name=field)
                field_cache[field] = aligned_df
            else:
                field_cache[field] = raw_df

    # === 【效能評估】階段 1 結束、階段 2 開始 ===
    t_parse_end = time.time()
    t_expand_start = time.time()

    # === 【優化後】使用帶對齊檢查的函式 ===
    p1p2_masks = f_factor(P1_conditions, field_cache, price_index=price_index)
    final_masks = c_factor(p1p2_masks, P3_conditions, field_cache, price_index=price_index)

    # === 【優化後】建立 V mask（帶對齊檢查與快取，受 USE_CACHE 控制） ===
    v_mask = build_v_pe_mask(data, pe_field="report:pe", window=4, price_index=price_index, use_cache=USE_CACHE)
    final_masks = expand_with_v(final_masks, v_mask, price_index=price_index)

    final_masks_by_json[label] = final_masks  

    # === 【效能評估】階段 2 結束 ===
    t_expand_end = time.time()

    # report_collection = sim_conditions(
    #     conditions=final_masks,
    #     resample='M',
    #     data=data
    # )
    # report_collections_by_json[label] = report_collection
    
    # save_all_for_label(
    #     report_collection=report_collection,
    #     base_pickle_dir=PICKLE_DIR,
    #     base_artifacts_dir=ART_DIR,
    #     label=Path(jp).stem,        
    #     to_parquet=True              
    # )

# === 【效能評估】記錄階段時間 ===

# === 【效能評估】記錄階段時間 ===
if json_files:
    timings['條件解析'] = t_parse_end - t_parse_start
    timings['批次展開'] = t_expand_end - t_expand_start
    strategy_count['展開後'] = sum(len(m) for m in final_masks_by_json.values())
    print(f"\n[Timer] 條件解析: {timings['條件解析']:.2f}s")
    print(f"[Timer] 批次展開: {timings['批次展開']:.2f}s")
    print(f"[Timer] 展開後策略總數: {strategy_count['展開後']}")
else:
    print("\n[Timer] 未處理任何 JSON（json_files 為空），略過計時統計")

交易日索引已取得：5943 個交易日，範圍 2000-01-04 ~ 2023-12-29

=== fcv_experiment_spec / P1 ===
- ROE_<5
- ROE_5_10
- ROE_10_15
- ROE_15_20
- ROE_>=20
- EPS_<0
- EPS_0_2
- EPS_>=2
- FCF_P_<0
- FCF_P_0_0.05
- FCF_P_>=0.05
- DEBTRATIO_0_0.35
- DEBTRATIO_0.35_0.55
- DEBTRATIO_>=0.55

=== fcv_experiment_spec / P3 ===
- C1_ROE_SEQ_qmax4
- C2_ROE_SEQ_riseq1
- C3_ROE_SEQ_yoy
- C4_REVENUE_GROWTH_qmax5
- C5_REVENUE_GROWTH_yoy
- C6_EPS_QOQ_yoy
- C7_EPS_QOQ_ytdavg_gt_lyytdavg
- C8_FCF_MOM_riseq2


C:\Users\dowgojashan\AppData\Local\Temp\ipykernel_26460\1282948242.py:53: UserWarning: [align_to_trading_days] report:roe 最早日期 (2000-05-15) 晚於價格最早交易日 (2000-01-04)，請檢查 adjust_index_of_report 規則。
  warnings.warn(
C:\Users\dowgojashan\AppData\Local\Temp\ipykernel_26460\1282948242.py:53: UserWarning: [align_to_trading_days] report:fcf_p 最早日期 (2000-05-15) 晚於價格最早交易日 (2000-01-04)，請檢查 adjust_index_of_report 規則。
  warnings.warn(
C:\Users\dowgojashan\AppData\Local\Temp\ipykernel_26460\1282948242.py:53: UserWarning: [align_to_trading_days] report:eps 最早日期 (2000-05-15) 晚於價格最早交易日 (2000-01-04)，請檢查 adjust_index_of_report 規則。
  warnings.warn(
C:\Users\dowgojashan\AppData\Local\Temp\ipykernel_26460\3441212717.py:62: UserWarning: [build_masks] DEBTRATIO_0_0.35 的 mask 索引未對齊到交易日。原始索引範圍: nan ~ nan, 交易日範圍: 2000-01-04 00:00:00 ~ 2023-12-29 00:00:00
  warnings.warn(
C:\Users\dowgojashan\AppData\Local\Temp\ipykernel_26460\3441212717.py:62: UserWarning: [build_masks] DEBTRATIO_0.35_0.55 的 mask 索引未對齊到交易日。原始索引範圍:

條件組合：P1 = ROE_<5，P2 = None（符合筆數：8144477）
條件組合：P1 = ROE_<5，P2 = EPS_<0（符合筆數：1908513）
條件組合：P1 = ROE_<5，P2 = EPS_0_2（符合筆數：5567946）
條件組合：P1 = ROE_<5，P2 = EPS_>=2（符合筆數：667575）
條件組合：P1 = ROE_<5，P2 = FCF_P_<0（符合筆數：2860551）
條件組合：P1 = ROE_<5，P2 = FCF_P_0_0.05（符合筆數：3760023）
條件組合：P1 = ROE_<5，P2 = FCF_P_>=0.05（符合筆數：1220002）
條件組合：P1 = ROE_<5，P2 = DEBTRATIO_0_0.35（符合筆數：0.0）
條件組合：P1 = ROE_<5，P2 = DEBTRATIO_0.35_0.55（符合筆數：0.0）
條件組合：P1 = ROE_<5，P2 = DEBTRATIO_>=0.55（符合筆數：0.0）
條件組合：P1 = ROE_5_10，P2 = None（符合筆數：227463）
條件組合：P1 = ROE_5_10，P2 = EPS_<0（符合筆數：147）
條件組合：P1 = ROE_5_10，P2 = EPS_0_2（符合筆數：208881）
條件組合：P1 = ROE_5_10，P2 = EPS_>=2（符合筆數：18435）
條件組合：P1 = ROE_5_10，P2 = FCF_P_<0（符合筆數：51281）
條件組合：P1 = ROE_5_10，P2 = FCF_P_0_0.05（符合筆數：112154）
條件組合：P1 = ROE_5_10，P2 = FCF_P_>=0.05（符合筆數：64028）
條件組合：P1 = ROE_5_10，P2 = DEBTRATIO_0_0.35（符合筆數：0.0）
條件組合：P1 = ROE_5_10，P2 = DEBTRATIO_0.35_0.55（符合筆數：0.0）
條件組合：P1 = ROE_5_10，P2 = DEBTRATIO_>=0.55（符合筆數：0.0）
條件組合：P1 = ROE_10_15，P2 = None（符合筆數：115743）
條件組合：P1 = ROE_10_15，P

C:\Users\dowgojashan\AppData\Local\Temp\ipykernel_26460\3441212717.py:62: UserWarning: [build_masks] C4_REVENUE_GROWTH_qmax5 的 mask 索引未對齊到交易日。原始索引範圍: nan ~ nan, 交易日範圍: 2000-01-04 00:00:00 ~ 2023-12-29 00:00:00
  warnings.warn(
C:\Users\dowgojashan\AppData\Local\Temp\ipykernel_26460\3441212717.py:62: UserWarning: [build_masks] C5_REVENUE_GROWTH_yoy 的 mask 索引未對齊到交易日。原始索引範圍: nan ~ nan, 交易日範圍: 2000-01-04 00:00:00 ~ 2023-12-29 00:00:00
  warnings.warn(


條件組合：P1+P2 = ROE_<5__None，P3 = None（符合筆數：8144477）
條件組合：P1+P2 = ROE_<5__None，P3 = C1_ROE_SEQ_qmax4（符合筆數：2108199）
條件組合：P1+P2 = ROE_<5__None，P3 = C2_ROE_SEQ_riseq1（符合筆數：3633998）
條件組合：P1+P2 = ROE_<5__None，P3 = C3_ROE_SEQ_yoy（符合筆數：3378671）
條件組合：P1+P2 = ROE_<5__None，P3 = C4_REVENUE_GROWTH_qmax5（符合筆數：0.0）
條件組合：P1+P2 = ROE_<5__None，P3 = C5_REVENUE_GROWTH_yoy（符合筆數：0.0）
條件組合：P1+P2 = ROE_<5__None，P3 = C6_EPS_QOQ_yoy（符合筆數：3613386）
條件組合：P1+P2 = ROE_<5__None，P3 = C7_EPS_QOQ_ytdavg_gt_lyytdavg（符合筆數：3666655）
條件組合：P1+P2 = ROE_<5__None，P3 = C8_FCF_MOM_riseq2（符合筆數：3553436）
條件組合：P1+P2 = ROE_<5__EPS_<0，P3 = None（符合筆數：1908513）
條件組合：P1+P2 = ROE_<5__EPS_<0，P3 = C1_ROE_SEQ_qmax4（符合筆數：253422）
條件組合：P1+P2 = ROE_<5__EPS_<0，P3 = C2_ROE_SEQ_riseq1（符合筆數：631367）
條件組合：P1+P2 = ROE_<5__EPS_<0，P3 = C3_ROE_SEQ_yoy（符合筆數：548696）
條件組合：P1+P2 = ROE_<5__EPS_<0，P3 = C4_REVENUE_GROWTH_qmax5（符合筆數：0.0）
條件組合：P1+P2 = ROE_<5__EPS_<0，P3 = C5_REVENUE_GROWTH_yoy（符合筆數：0.0）
條件組合：P1+P2 = ROE_<5__EPS_<0，P3 = C6_EPS_QOQ_yoy（符合筆數：524487）
條件組合：P

In [18]:
# === 額外保險：最終對齊檢查 ===
# 前面的展開流程已完成：
# 1. 使用 align_to_trading_days() 統一對齊所有資料
# 2. 在 build_masks/f_factor/c_factor/expand_with_v 中都加入了對齊檢查
# 3. 最終的 final_masks 應該已經全部對齊到 price_index
#
# 此 cell 的 fix_position_df 函式作為額外安全檢查。

import pandas as pd

# 使用前面流程已取得的 price_index（避免重複取得）
px_idx = price_index if 'price_index' in globals() else data.get("price:close").index

def fix_position_df(df, target_index):
    """
    把任意 DataFrame/Series 的索引轉成 DatetimeIndex，排序、去重，
    再對齊到 target_index（用 ffill 延續部位），最後轉成 float。
    
    注意：此函式主要作為額外檢查，因為前面流程已做過對齊。
    """
    # 轉 DataFrame
    if isinstance(df, pd.Series):
        df = df.to_frame()
    elif not isinstance(df, pd.DataFrame):
        raise TypeError(f"position 必須是 DataFrame/Series，拿到的是 {type(df)}")

    # 索引轉 DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        idx = pd.to_datetime(df.index, errors="coerce")
        df = df.loc[idx.notna()].copy()
        df.index = idx[idx.notna()]

    # 去重 & 排序
    if df.index.has_duplicates:
        df = df[~df.index.duplicated(keep="last")]
    df = df.sort_index()

    # 對齊交易日；用 ffill 讓訊號/部位延續到下一個可交易日
    df = df.reindex(target_index, method="ffill")

    # True/False -> 1.0/0.0，或原本權重→float
    try:
        df = df.astype("float32")
    except Exception:
        df = df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    # 沒有部位的日子補 0
    df = df.fillna(0.0)

    return df

# === 【額外安全檢查】對所有 cached masks 做一次最終修正 ===
# 此檢查可以發現是否有遺漏的對齊問題
needed_fix = False
for label, masks in final_masks_by_json.items():
    for name, pos in masks.items():
        if not isinstance(pos.index, pd.DatetimeIndex) or not pos.index.equals(px_idx):
            needed_fix = True
            break
    if needed_fix:
        break

if needed_fix:
    print("發現部分 masks 未完全對齊，進行修正...")
    fixed_masks_by_json = {}
    for label, masks in final_masks_by_json.items():
        fixed = {}
        for name, pos in masks.items():
            fixed[name] = fix_position_df(pos, px_idx)
        fixed_masks_by_json[label] = fixed
    final_masks_by_json = fixed_masks_by_json
    print("所有 cached masks 已轉為 DatetimeIndex 並對齊交易日。")
else:
    print("所有 cached masks 已正確對齊到交易日（無需額外修正）。")


所有 cached masks 已正確對齊到交易日（無需額外修正）。


In [19]:
import numpy as np
import pandas as pd

# === 【效能評估】階段 3：策略預篩計時開始 ===
t_prefilter_start = time.time()

# === 參數：最低交易次數與最多回測策略數（可選） ===
MIN_TRADES = 5     # 最低進場次數門檻（依資料期間長短調整）
TOP_K      = 10000    # 只跑前 K 個（依 trade 次數排序），設 None 表示不限制
ENTRY_THRESH = 1e-9  # 權重/持有的判定門檻（>0 視為持有；避免浮點雜訊）

def count_trades_from_position(pos: pd.DataFrame, thresh=ENTRY_THRESH) -> int:
    """
    以遮罩/權重矩陣估算「總進場數」。
    對每一檔股票：當天 (pos>thresh) 且 昨天 (pos<=thresh) 視為進場一次。
    回傳全欄位加總（多資產求和）。
    """
    if isinstance(pos, pd.Series):
        pos = pos.to_frame()
    held    = pos.values > thresh
    held_y  = np.vstack([np.zeros((1, held.shape[1]), dtype=bool), held[:-1, :]])
    entries = held & (~held_y)
    return int(entries.sum())

def quick_stats(pos):
    import pandas as pd

    # 1) 規格化索引 → DatetimeIndex（無害、最保險）
    if not isinstance(pos.index, pd.DatetimeIndex):
        pos = pos.copy()
        pos.index = pd.to_datetime(pos.index, errors="coerce")
    idx = pos.index

    # 2)（可選）若可能有未排序的情況，確保排序
    if not idx.is_monotonic_increasing:
        pos = pos.sort_index()
        idx = pos.index

    # 3) 基本統計
    trades = int((pos.diff().abs() > 0).any(axis=1).sum())
    days = max((idx[-1] - idx[0]).days + 1, 1)
    active = int((pos.abs().sum(axis=1) > 0).sum())
    span_yrs = max((idx[-1] - idx[0]).days / 365.25, 1e-9)

    # 4) 這裡不要再 .date()，改用 Timestamp 的 .date() 或 strftime 都行
    return dict(
        trades_total=trades,
        trades_per_year=trades / span_yrs,
        active_days=active,
        coverage_ratio=active / days,
        start=idx[0].date().isoformat(),   # or idx[0].strftime("%Y-%m-%d")
        end=idx[-1].date().isoformat(),    # or idx[-1].strftime("%Y-%m-%d")
    )


# === 對 600+ 策略先做預篩 ===
prefilter_report = []   # 存放預篩統計供檢視
filtered_final_masks_by_json = {}  # 只留下合格的策略

for label, masks in final_masks_by_json.items():
    kept = {}
    for name, pos in masks.items():
        stats = quick_stats(pos)
        prefilter_report.append((label, name, stats["trades_total"], stats["trades_per_year"],
                                 stats["coverage_ratio"], stats["start"], stats["end"]))
        if stats["trades_total"] >= MIN_TRADES:
            kept[name] = pos
    # 依照 TOP_K 限制數量（以交易次數排序）
    if TOP_K is not None and kept:
        # 重新排序挑前 K
        ranked = sorted(kept.items(),
                        key=lambda kv: count_trades_from_position(kv[1]),
                        reverse=True)[:TOP_K]
        kept = dict(ranked)
    if kept:
        filtered_final_masks_by_json[label] = kept

# 簡短總覽：即將回測的策略數
total_before = sum(len(m) for m in final_masks_by_json.values())
total_after  = sum(len(m) for m in filtered_final_masks_by_json.values())
print(f"將回測 {total_after} / {total_before} 個策略（門檻 MIN_TRADES={MIN_TRADES}, TOP_K={TOP_K}）")

# （可選）印出前幾筆預篩統計
for row in prefilter_report[:10]:
    print("label=",row[0], "name=",row[1], 
          "trades=",row[2], "trades/yr=",f"{row[3]:.1f}",
          "cover=",f"{row[4]*100:.1f}%","span=",row[5],"~",row[6])

# === 【效能評估】階段 3 結束 ===
t_prefilter_end = time.time()
timings['策略預篩'] = t_prefilter_end - t_prefilter_start
strategy_count['預篩後'] = total_after
print(f"\n[Timer] 策略預篩: {timings['策略預篩']:.2f}s")


將回測 1118 / 2844 個策略（門檻 MIN_TRADES=5, TOP_K=10000）
label= fcv_experiment_spec name= ROE_<5__None__None__v0 trades= 64 trades/yr= 2.7 cover= 66.8% span= 2000-01-04 ~ 2023-12-29
label= fcv_experiment_spec name= ROE_<5__None__None__v1 trades= 91 trades/yr= 3.8 cover= 64.1% span= 2000-01-04 ~ 2023-12-29
label= fcv_experiment_spec name= ROE_<5__None__C1_ROE_SEQ_qmax4__v0 trades= 86 trades/yr= 3.6 cover= 64.1% span= 2000-01-04 ~ 2023-12-29
label= fcv_experiment_spec name= ROE_<5__None__C1_ROE_SEQ_qmax4__v1 trades= 88 trades/yr= 3.7 cover= 61.2% span= 2000-01-04 ~ 2023-12-29
label= fcv_experiment_spec name= ROE_<5__None__C2_ROE_SEQ_riseq1__v0 trades= 88 trades/yr= 3.7 cover= 65.8% span= 2000-01-04 ~ 2023-12-29
label= fcv_experiment_spec name= ROE_<5__None__C2_ROE_SEQ_riseq1__v1 trades= 91 trades/yr= 3.8 cover= 64.1% span= 2000-01-04 ~ 2023-12-29
label= fcv_experiment_spec name= ROE_<5__None__C3_ROE_SEQ_yoy__v0 trades= 85 trades/yr= 3.5 cover= 63.8% span= 2000-01-04 ~ 2023-12-29
label= fcv_expe

In [20]:
import pandas as pd

def _ensure_dtindex(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.copy()
        df.index = pd.to_datetime(df.index, errors="coerce")
    if df.index.hasnans:
        bad_cnt = int(df.index.isna().sum())
        raise ValueError(f"Index contains {bad_cnt} NaT after to_datetime — 檢查你的日期欄位/格式。")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    return df

# 在回測 for-loop 之前先執行這段就地淨化
for label, masks in list(filtered_final_masks_by_json.items()):
    filtered_final_masks_by_json[label] = {
        name: _ensure_dtindex(pos) for name, pos in masks.items()
    }

# （可選）快速自我檢查，確認真的都是 DatetimeIndex
any_bad = [
    (label, name) for label, masks in filtered_final_masks_by_json.items()
    for name, pos in masks.items() if not isinstance(pos.index, pd.DatetimeIndex)
]
assert not any_bad, f"仍有索引不是 DatetimeIndex: {any_bad[:3]} ..."


In [21]:
# === 【效能評估】階段 4 與階段 5：回測執行 + 結果儲存 ===
t_backtest_total = 0.0
t_save_total = 0.0

for label, final_masks in filtered_final_masks_by_json.items():
    print(f"\n=== Backtesting (prefiltered): {label} | {len(final_masks)} strategies ===")
    
    _t = time.time()
    report_collection = sim_conditions(
        conditions=final_masks,
        resample="M",
        data=data
    )
    report_collections_by_json[label] = report_collection
    t_backtest_total += time.time() - _t

    _t = time.time()
    save_all_for_label(
        report_collection=report_collection,
        base_pickle_dir=PICKLE_DIR,
        base_artifacts_dir=ART_DIR,
        label=label,
        to_parquet=True
    )
    t_save_total += time.time() - _t

timings['回測執行'] = t_backtest_total
timings['結果儲存'] = t_save_total
print(f"\n[Timer] 回測執行: {timings['回測執行']:.2f}s")
print(f"[Timer] 結果儲存: {timings['結果儲存']:.2f}s")



=== Backtesting (prefiltered): fcv_experiment_spec | 1118 strategies ===


Backtesting progress: 100%|██████████| 1118/1118 [19:48<00:00,  1.06s/condition] 


[OK] Pickle saved → results_pickle\fcv_experiment_spec.pkl


d:\研究所\stock_factor_lab-master\stock_factor_lab\.venv\lib\site-packages\pandas\io\parquet.py:159: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
d:\研究所\stock_factor_lab-master\stock_factor_lab\.venv\lib\site-packages\pandas\io\parquet.py:159: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
d:\研究所\stock_factor_lab-master\stock_factor_lab\.venv\lib\site-packages\pandas\io\parquet.py:159: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
d:\研究所\stock_factor_lab-master\stock_factor_lab\.venv\lib\site-packages\pandas\io\parquet.py:159: UserWarning: The DataFrame has column names of mixed 

[OK] stats saved → results_artifacts\fcv_experiment_spec
[OK] Artifacts exported → results_artifacts\fcv_experiment_spec

[Timer] 回測執行: 1194.84s
[Timer] 結果儲存: 160.96s


d:\研究所\stock_factor_lab-master\stock_factor_lab\.venv\lib\site-packages\pandas\io\parquet.py:159: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


# **效能評估彙總**

In [22]:
# === 效能評估：彙總輸出 ===
total_end = time.time()
total_elapsed = total_end - total_start
timings['總計'] = total_elapsed

print("\n" + "="*60)
print(f"批次回測各階段耗時統計（USE_CACHE = {USE_CACHE}）")
print("="*60)
STAGE_ORDER = ['條件解析', '批次展開', '策略預篩', '回測執行', '結果儲存']
for stage in STAGE_ORDER:
    if stage in timings:
        elapsed = timings[stage]
        pct = elapsed / total_elapsed * 100
        print(f"  {stage:8s}: {elapsed:>10.2f}s  ({pct:>5.1f}%)")
print(f"  {'總計':8s}: {total_elapsed:>10.2f}s  (100.0%)")

print(f"\n硬體規格：")
for k, v in hardware_info.items():
    print(f"  {k}: {v}")

print(f"\n策略數量：")
for k, v in strategy_count.items():
    print(f"  {k}: {v}")

# === 自動寫入 JSON 檔案（效能彙總）===
output_log = {
    "use_cache": USE_CACHE,
    "timings": timings,
    "strategy_count": strategy_count,
    "hardware_info": hardware_info,
}
TIMING_LOG_DIR = _Path_for_timing("timing_logs")
TIMING_LOG_DIR.mkdir(parents=True, exist_ok=True)
log_filename = TIMING_LOG_DIR / f"timing_log_cache{USE_CACHE}_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(log_filename, "w", encoding="utf-8") as f:
    _json_for_timing.dump(output_log, f, ensure_ascii=False, indent=2)
print(f"\n[Timer] 計時記錄已儲存至：{log_filename}")



批次回測各階段耗時統計（USE_CACHE = True）
  條件解析    :       0.52s  (  0.0%)
  批次展開    :     226.60s  (  8.4%)
  策略預篩    :     698.99s  ( 26.0%)
  回測執行    :    1194.84s  ( 44.4%)
  結果儲存    :     160.96s  (  6.0%)
  總計      :    2690.51s  (100.0%)

硬體規格：
  timestamp: 2026-07-01T21:07:05
  USE_CACHE: True
  os: Windows-10-10.0.26200-SP0
  python: 3.10.11
  machine: AMD64
  processor: Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
  cpu_count_physical: 20
  cpu_count_logical: 28
  ram_gb: 63.7
  cpu_freq_max_mhz: 2100.0

策略數量：
  展開後: 2844
  預篩後: 1118

[Timer] 計時記錄已儲存至：timing_logs\timing_log_cacheTrue_20260701_215155.json


In [23]:
# === F[1] 條件重複使用次數統計 ===
# 統計同一 F[1] 條件在多少組策略中重複出現
from collections import Counter

p1_counter = Counter()
for label, masks in final_masks_by_json.items():
    for combo_key in masks.keys():
        # combo_key 格式: P1_name__P2_name__P3_name__V
        p1_name = combo_key.split('__')[0]
        p1_counter[p1_name] += 1

print("F[1] 條件重複使用次數（前 15 名）：")
for p1_name, count in p1_counter.most_common(15):
    print(f"  {p1_name}: {count} 組策略")

if p1_counter:
    avg_repeat = sum(p1_counter.values()) / len(p1_counter)
    max_repeat = p1_counter.most_common(1)[0][1]
    print(f"\n平均每個 F[1] 條件被使用 {avg_repeat:.1f} 組")
    print(f"最高使用次數：{max_repeat} 組")
    print(f"P[1] 條件總數: {len(p1_counter)}")

# 把這個結果也寫進 timing_log（補充）
import json as _j
from pathlib import Path as _P
_log_files = sorted(_P('timing_logs').glob(f'timing_log_cache{USE_CACHE}_*.json'))
if _log_files:
    _latest = _log_files[-1]
    with open(_latest, 'r', encoding='utf-8') as f:
        _data = _j.load(f)
    _data['p1_repeat_count'] = dict(p1_counter)
    _data['p1_repeat_summary'] = {
        'avg_repeat': avg_repeat if p1_counter else 0,
        'max_repeat': max_repeat if p1_counter else 0,
        'unique_p1_count': len(p1_counter),
    }
    with open(_latest, 'w', encoding='utf-8') as f:
        _j.dump(_data, f, ensure_ascii=False, indent=2)
    print(f"\n[Timer] F[1] 重複統計已附加至：{_latest}")


F[1] 條件重複使用次數（前 15 名）：
  EPS_<0: 216 組策略
  EPS_0_2: 216 組策略
  EPS_>=2: 216 組策略
  FCF_P_<0: 216 組策略
  FCF_P_0_0.05: 216 組策略
  FCF_P_>=0.05: 216 組策略
  DEBTRATIO_0_0.35: 216 組策略
  DEBTRATIO_0.35_0.55: 216 組策略
  DEBTRATIO_>=0.55: 216 組策略
  ROE_<5: 180 組策略
  ROE_5_10: 180 組策略
  ROE_10_15: 180 組策略
  ROE_15_20: 180 組策略
  ROE_>=20: 180 組策略

平均每個 F[1] 條件被使用 203.1 組
最高使用次數：216 組
P[1] 條件總數: 14

[Timer] F[1] 重複統計已附加至：timing_logs\timing_log_cacheTrue_20260701_215155.json
